# 노드 B — tonight

**6 GPU = 3노드 x 2 GPU. 이 노트북은 노드 B 전용.**

세 노드 전부 `exp5_tonight.py` 하나만 부른다. 잡 정의 · 우선순위 · 스킵 · 집계가 전부 거기 있다.
**세 노트북은 내용이 동일하고 노드 문자만 다르다** — 학습으로 배정되든 eval로 배정되든 이 하나로 된다.

**예산**: eval 1셀(LIBERO-10 x 50ep/task = 500ep) ≈ 2 GPU-h · 학습 1잡(150k) ≈ 8 GPU-h
(2026-08-24 밤 실측: `bimamba_pure` 등 4잡을 2 GPU로 2배치, 총 ~16h)

---

## 흐름

1. 부팅 → 2. 인벤토리 → 3. **역할 배정** → 4. 계획 → 5. preflight(한 번만)
→ **6. eval** 또는 **7. 학습** (3번이 정해준 쪽만) → 8. 이어서 → 집계

6)·7) 셀은 배정된 역할이 아니면 스스로 건너뛴다. 잘못 눌러도 사고 안 난다.

## 우선순위 (`exp5_tonight._all_jobs`)

1. **TE 축** (stride=1 + coeff 0.01) — 크로스오버 K 찍기. **TE는 재학습이 필요 없다**
   (ACT ckpt에 플래그만 얹음). K=100은 이미 측정됨: ACT+TE 30.5 vs BiMamba+TE 49.3 = **+18.8**.
   K=50에서 ACT+TE가 이기고 K=100에서 뒤집히면 그게 논문 그림 1이다.
2. **K=100 레짐 맵**, 긴 stride부터 (8/18 가설: BiMamba는 긴 stride에서 산다)
3. **K=50 레짐 맵** (8/18 가설: carry는 짧은 stride에서 산다)
4. **순수 BiMamba**(`bimamba_pure`) 셀 — 8/24 밤에 K=100/50/150 학습 완료

## 학습 큐

인벤토리에서 `MISS`/`PART`인 태그가 자동으로 들어간다. 순서는 `TRAIN_PRIORITY` 고정 목록 →
그 뒤는 **막힌 eval 셀 수** 순. 이미 학습된 건 빠지고, 중단된 것(PART)은 resume된다.

`bimamba_pure` = `use_chunk_pairs=False` + `sscp_enabled=false`. 기존 `bimamba` 태그는
carry-on + chunk-pair로 학습돼 있어서 eval에서 carry만 꺼도 순수 BiMamba가 아니다
(은지님 8/18 지적). 오염된 프록시는 `bimamba_cpoff`로 라벨에 명시해 표에 남긴다.


## 1) 부팅

In [ ]:
import sys
from pathlib import Path

_h = Path.cwd()
_r = next(c for c in (_h, *_h.parents) if (c / 'notebooks' / 'libero' / 'exp5_tonight.py').exists())
for _p in (_r / 'notebooks', _r / 'notebooks' / 'libero'):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
import exp5_tonight as X
X = importlib.reload(X)
cf, v23 = X.setup()          # common_final reload + 태그 등록 (순서 중요)

NODE = 'B'
# 이 노드에서 쓸 GPU. 한 노드에서 창을 2개 띄울 때만 [2, 3] 처럼 직접 지정.
GPUS = v23.available_gpus([0, 1])
print('NODE', NODE, '| GPUS', GPUS)


## 2) 인벤토리 — 뭐가 학습돼 있고 뭐가 없나

서버 파일시스템을 실제로 스캔한다. `MISS` = 학습 필요, `PART` = 중단됨(resume 대상).
`MISS`/`PART`인 태그를 참조하는 eval 셀은 자동으로 큐에서 빠진다.

In [ ]:
rows = X.inventory()


## 3) 역할 배정 ← **여기가 오늘 뭘 할지 정한다**

규칙: **ready eval 큐를 먼저 포화시키고, 남는 노드만 학습에 준다.**
한 노드(2 GPU)의 하룻밤은 eval ~12셀 vs 학습 2잡인데, 학습 2잡이 나중에 열어주는 eval은
보통 몇 셀뿐이라 **지금 돌릴 eval이 쌓여 있으면 eval이 거의 항상 이긴다.**

| ready eval | 배정 |
|---|---|
| ≥25셀 | A·B·C 전부 eval |
| 13~24셀 | B·C = eval, A = 학습 |
| 1~12셀 | B = eval, A·C = 학습 |
| 0셀 | A·B·C 전부 학습 |
| 학습 큐 0 | 전부 eval |

출력이 "기본값과 다르다"면 알려주는 `X.ROLE_OVERRIDE = ...` 한 줄을 **세 노트북 모두**에 넣어야 잡이 안 겹친다.

In [ ]:
roles = X.suggest()

# 위 출력이 "기본값과 다르다" 라고 하면, 알려주는 한 줄을 **세 노트북 모두**에 붙여넣고
# 이 셀부터 다시 실행할 것. (세 노드가 같은 역할표를 봐야 잡이 안 겹친다)
# X.ROLE_OVERRIDE = {'A': 'eval'}


## 4) 계획 — 이 노드 몫

실행 전에 목록을 눈으로 확인할 것.

In [ ]:
plan = X.plan(NODE, GPUS)


## 5) preflight (약 12분) — **셋 중 한 노드에서 한 번만**

`--policy.n_action_steps` / `--policy.temporal_ensemble_coeff` override가 `lerobot_eval`에서 실제로 먹는지 확인한다. 여기서 죽으면 override 경로가 막힌 것이고, 그러면 조합별 학습이 필요해져 **계획을 전면 수정**해야 한다.

다른 노드에서 이미 통과했으면 건너뛸 것.

In [ ]:
X.preflight(gpu=GPUS[0], n_ep=5)


## 6) eval — 3)에서 `eval`로 배정됐을 때

먼저 dry-run으로 커맨드를 보고 실행. `GPUS` 수만큼 청크로 돌고 청크마다 블로킹한다.
로그는 `outputs/final/_logs/exp5__*.log`. **아침에 다시 실행하면 완료분은 skip되고 이어서 돈다.**

In [ ]:
X.run_evals(plan['eval'][:2], plan['gpus'], dry=True)


In [ ]:
if X.role_of(NODE) != 'eval':
    print('노드 ' + NODE + ' 는 train 으로 배정됐다 -> 7)번 학습 셀을 쓸 것. 여기는 건너뛴다.')
else:
    X.run_evals(plan['eval'], plan['gpus'])


## 7) 학습 — 3)에서 `train`으로 배정됐을 때

**dry-run에서 반드시 확인**: `bimamba_pure` 커맨드에 `--use_chunk_pairs`가 **없고** `--policy.sscp_enabled=false`가 **있어야** 한다. 이거 하나 틀리면 8시간을 날린다.

실행 셀은 잡이 끝날 때까지(~8h/잡) 블로킹한다.

In [ ]:
X.run_trains(plan['train'], plan['gpus'], dry=True)


In [ ]:
if X.role_of(NODE) != 'train':
    print('노드 ' + NODE + ' 는 eval 로 배정됐다 -> 6)번 eval 셀을 쓸 것. 여기는 건너뛴다.')
elif not plan['train']:
    print('학습 큐가 비었다.')
else:
    X.run_trains(plan['train'], plan['gpus'])


## 8) 이어서 — 다음 배치

위 배치가 끝나면 인벤토리가 바뀐다. 이 셀로 역할·계획을 다시 뽑고 6) 또는 7)로 돌아간다.

In [ ]:
# 위 배치가 끝나면 인벤토리가 바뀐다. 이 셀로 역할/계획을 다시 뽑고 6) 또는 7)로 돌아간다.
X = importlib.reload(X)
cf, v23 = X.setup(verbose=False)
roles = X.suggest()
plan = X.plan(NODE, GPUS)


## 아침에 볼 것

`X.report()` 가 TE 표 + 레짐 맵을 전부 찍는다. 판단 기준:

| 결과 | 다음 |
|---|---|
| `act+TE@K50` **<** `bimamba+TE@K50` | 크로스오버가 K<50 → ACT K=15/10 학습 필요 |
| `act+TE@K50` **>** `bimamba+TE@K50` | **크로스오버 = K 50~100 확정.** 논문을 "long-chunk regime"으로 리라이트 시작 |
| 레짐 맵에서 `bimamba`(순수)가 긴 stride에서 `act` 상회 | 그 지점으로 seed 1,2 추가 |
| `carry`가 짧은 stride에서 `act` 상회 | 레짐 논문 확정 ("보완재가 아니라 대체재") |
| 아무것도 ACT를 못 이김 | TE 축이 유일한 카드 → 거기로 올인 |

**주의**: seed 1개 · 500ep 기준 binomial SE ≈ ±2.2%p. **5%p 미만 차이는 주장하지 말 것.**

In [ ]:
X.report()
